# S&P 500 timing with TimesFM forecasts -- backtest

**Idea.** Each month, ask Google's TimesFM 2.5 time-series foundation model (already used by the app's `forecast_quantiles` tool) for the distribution of SPY's return **one period ahead** (to the next month-end). Hold SPY when the forecast is favourable, T-bills otherwise.

**Rules** (all computed from the same cached forecasts; $R$ = forecast return to the next month-end, $c$ = T-bill return over the same period, known at the decision):

| Rule | Risk-on (hold SPY) when ... |
|---|---|
| **R1 median** | forecast median $q_{50}(R) > c$ |
| **R2 probability** | forecast $P(R > c) \ge$ `P_STAR` (interpolated from the forecast deciles) |
| **R3 tail** | forecast 10th percentile $q_{10}(R) \ge$ `TAIL_Q10` (avoid months with a fat forecast left tail) |
| **R4 median + trend** | R1 **and** close > SMA200 |

**Comparisons**: SMA200 monthly (the rule from `../sma/`), ARIMA(1,1,1) forecasts (same walk-forward, rule R1), 1-month momentum (trailing month return > T-bills), buy & hold, T-bills, and a random-timing placebo.

**Implementation**

| Item | Choice |
|---|---|
| Decision date $d$ | last trading day of each month (`monthly` rebalancing, as in the SMA200 study) |
| Model input | the last `CONTEXT` daily closes of `INPUT_FIELD` up to and including $d$ -- nothing after $d$ |
| Forecast | TimesFM 2.5 200M zero-shot, deciles $q_{10}..q_{90}$, step $k$ = business days from $d$ to the next month-end |
| Execution | default `next_open`: decided at close $d$, filled at the open of $d+1$; returns open-to-open (same engine as the SMA backtest) |
| Equity returns | total return (dividends reinvested) |
| Risk-off asset | `irx`: accrual at the 13-week T-bill yield; `bil`: BIL ETF (2007+); `zero`: cash |
| Costs | `COST_BPS` per switch |

The notebook first scores the **forecasts themselves** (direction, ranking, calibration, accuracy vs a random walk and ARIMA) and only then the trading rules, so a lucky backtest cannot hide a forecast with no skill.

> Decision-support / research only -- this notebook never places orders. Past performance is not indicative of future results. TimesFM may have seen S&P 500 history during pre-training, so results before its release are not truly out-of-sample (section 7).

In [1]:
# ---- Parameters -------------------------------------------------------------
EQUITY_TICKER = "SPY"     # investable S&P 500 ETF (as in the SMA notebooks)
CASH_SOURCE = "irx"       # "irx" | "bil" | "zero"
REBALANCE = "monthly"     # fixed: one decision per month, forecast one month ahead
INPUT_FIELD = "tr_close"  # series fed to TimesFM: "tr_close" (dividend-adjusted, like the app's yfinance closes) | "close"
CONTEXT = 1024            # daily closes per forecast (the app's TimesFM context; ~4 years)
HORIZON_MAX = 23          # steps forecast at each decision; the step used = business days to the next month-end
START = "1997-04-01"      # first backtest date (the first decision needs CONTEXT bars of history)
END = None                # None = latest available
COST_BPS = 5.0            # cost per switch, bp of portfolio value
EXECUTION = "next_open"   # "next_open" (the rule) | "next_close" | "same_close" (look-ahead biased, comparison only)
MAIN_RULE = "R1"          # rule shown in the detailed charts: "R1" | "R2" | "R3" | "R4"
P_STAR = 0.55             # R2: minimum forecast probability that SPY beats T-bills over the month
TAIL_Q10 = -0.08          # R3: T-bills when the forecast 10th-percentile monthly return is below this
SMA_WINDOW = 200          # SMA benchmark and the R4 trend filter
MODEL_RELEASE = "2025-09-02"  # TimesFM 2.5 weights published on Hugging Face (pre-training data end is not published)
N_PLACEBO = 2000          # random-timing placebo draws
N_BOOT = 10_000           # block-bootstrap draws
BLOCK = 6                 # bootstrap block length (months)
NW_LAGS = 6               # Newey-West lags (months)
SEED = 42
REFRESH_DATA = False      # True = ignore the local price cache and re-download
CACHE_MAX_AGE_HOURS = 12
REFRESH_FORECASTS = False  # True = recompute all forecasts (~25 min TimesFM + ~10 min ARIMA on CPU)
RUN_CONTEXT_SWEEP = False  # True = also forecast with 256 / 512-bar context (cached; extra CPU time on first run)

In [2]:
import math
import sys
import time
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from scipy import stats

# Shared helpers: notebooks/backtest/bt_common.py
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
from bt_common import (  # noqa: E402
    CACHE_DIR, EXECUTIONS, REPO_ROOT, SESSION, backtest_signal, block_indices, cash_returns, check_days,
    decision_panel, newey_west, perf_stats, setup_plotly, sma_signal, strategy_stats, total_return_frame, trade_log,
)
from bt_common import load_history as _load_history  # noqa: E402

# The app's TimesFM loader (same weights and ForecastConfig as the agent tool), ARIMA and random-walk cones.
from investment_firm.core.tools import forecast_tools as ft  # noqa: E402
from investment_firm.data import arima_garch as ag  # noqa: E402
from investment_firm.data import forecast as fc  # noqa: E402

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
C_STRAT, C_BH, C_CASH, C_SMA, C_ARIMA, C_MOM = "#1f77b4", "#ff7f0e", "#2ca02c", "#333333", "#9467bd", "#8c564b"

# fig.show() emits plain HTML, so charts render in VS Code without the "Jupyter Notebook Renderers" extension.
PLOTLY_JS = "cdn"  # "cdn" = load plotly.js from cdn.plot.ly | "inline" = embed it (offline, ~4.6 MB per chart)
setup_plotly(PLOTLY_JS)
load_history = partial(_load_history, refresh=REFRESH_DATA, max_age_hours=CACHE_MAX_AGE_HOURS)
RNG = np.random.default_rng(SEED)

assert ft.timesfm_available(), 'TimesFM is not installed: .venv\\Scripts\\python.exe -m pip install -e ".[forecast]"'
print("repo root:", REPO_ROOT)
print("yfinance session:", type(SESSION).__name__ if SESSION is not None else "default")
print("TimesFM weights:", ft.forecast_model_repo(), "| device:", ft.forecast_device())

repo root: C:\Users\wn686\OneDrive - Deutsche Börse AG\Desktop\REPOs\investment-firm-agents
yfinance session: Session
TimesFM weights: google/timesfm-2.5-200m-pytorch | device: cpu


## 1. Data

In [3]:
px_all = total_return_frame(load_history(EQUITY_TICKER))
px_all = px_all[px_all["open"] > 0]
if END is not None:
    px_all = px_all.loc[:END]

irx = load_history("^IRX")["Close"].rename("irx_pct")
bil_px = total_return_frame(load_history("BIL")) if CASH_SOURCE == "bil" else None
CASH = {f: cash_returns(px_all.index, f, CASH_SOURCE, irx, bil_px) for f in ("tr_open", "tr_close")}
series = px_all[INPUT_FIELD]  # what the forecasting models see

print(f"{EQUITY_TICKER}: {px_all.index[0].date()} -> {px_all.index[-1].date()}  ({len(px_all):,} bars)")
print(f"^IRX: {irx.index[0].date()} -> {irx.index[-1].date()}, last = {irx.iloc[-1]:.2f}%")

SPY: 1993-01-29 -> 2026-10-19  (8,488 bars)
^IRX: 1960-01-04 -> 2026-10-19, last = 4.08%


## 2. Walk-forward forecasts

At every decision date $d$ (last trading day of the month) each model sees **only** the last `CONTEXT` closes up to and including $d$ and forecasts `HORIZON_MAX` daily steps. The step used is $k$ = number of business days (Mon-Fri) from $d$ to the next month-end. That is known at $d$; exchange holidays are ignored, so $k$ can exceed the real number of trading days by 0-2.

The forecast return quantiles are $q_j(R) = \hat P_{j}(d+k) / P_d - 1$, where $\hat P_j(d+k)$ is the model's $j$-th decile of the price $k$ steps ahead.

| Model | What it is |
|---|---|
| **TimesFM 2.5 200M** | zero-shot foundation model, loaded through the app's `forecast_tools` (`normalize_inputs`, flip invariance, `infer_is_positive`, quantile-crossing fix). Output index 0 is the mean, 1..9 the deciles q10..q90 |
| **ARIMA(1,1,1)** | Gaussian MLE on log prices (`data/arima_garch.py`), refitted at every decision on the same context |
| **Random walk** | driftless log-normal cone with trailing 252-day volatility (`data/forecast.py`): the "no information" baseline the app draws next to every TimesFM cone |

Forecasts are cached per decision date under `.cache/notebooks/timesfm/`, so only new month-ends are computed on later runs. The first run takes ~3-4 s per TimesFM forecast on CPU.

In [4]:
DECILES = np.round(np.arange(0.1, 0.91, 0.1), 1)
QCOLS = [f"q{int(round(q * 100))}" for q in DECILES]
FC_DIR = CACHE_DIR / "timesfm"
FC_DIR.mkdir(parents=True, exist_ok=True)


def cache_path(kind: str, context: int) -> Path:
    tag = ft.forecast_model_repo().split("/")[-1] if kind == "timesfm" else "arima111"
    return FC_DIR / f"{kind}_{EQUITY_TICKER.replace('^', 'IDX_')}_{INPUT_FIELD}_ctx{context}_h{HORIZON_MAX}_{tag}.pkl"


def walk_forward(kind: str, dates, context: int, refresh: bool = False, batch: int = 16) -> dict:
    # {decision date: (HORIZON_MAX, 9) array of decile price paths}; each forecast uses closes <= its date only.
    path = cache_path(kind, context)
    store = {} if refresh or not path.exists() else pd.read_pickle(path)
    todo = [d for d in dates if d not in store]
    if todo:
        print(f"{kind} (context {context}): {len(todo)} new forecasts, {len(store)} cached", flush=True)
        model = ft._load_model(ft.forecast_model_repo(), ft.forecast_device()) if kind == "timesfm" else None
        t0 = time.time()
        for i in range(0, len(todo), batch):
            chunk = todo[i : i + batch]
            ctx = [series.loc[:d].to_numpy(float)[-context:] for d in chunk]
            if kind == "timesfm":
                _, q = model.forecast(horizon=HORIZON_MAX, inputs=ctx)
                for d, arr in zip(chunk, np.asarray(q)[:, :HORIZON_MAX, 1:10]):
                    store[d] = arr.astype(float)
            else:
                for d, c in zip(chunk, ctx):
                    try:
                        r = ag.arima_garch_quantiles(c, HORIZON_MAX, garch=False, quantiles=DECILES)
                        store[d] = np.column_stack([r["quantiles"][k] for k in QCOLS])
                    except fc.ForecastError:
                        store[d] = np.full((HORIZON_MAX, len(QCOLS)), np.nan)
            pd.to_pickle(store, path)  # save progress after every batch
            print(f"  {min(i + batch, len(todo))}/{len(todo)}  {time.time() - t0:.0f}s", flush=True)
    return {d: store[d] for d in dates}


chk_all = check_days(px_all.index, REBALANCE)
CHECK_DATES = px_all.index[chk_all.to_numpy()]
n_hist = pd.Series(np.arange(1, len(px_all) + 1), index=px_all.index)
DECISIONS = CHECK_DATES[n_hist.reindex(CHECK_DATES).to_numpy() >= CONTEXT]
LAST = px_all.index[-1]
LIVE = not bool(chk_all.iloc[-1])  # latest close is not a month-end: also forecast from it (section 6)
FC_DATES = list(DECISIONS) + ([LAST] if LIVE else [])
assert DECISIONS[0] < pd.Timestamp(START), f"first decision {DECISIONS[0].date()} is after START; move START later"


def next_check(d: pd.Timestamp) -> pd.Timestamp:
    # Calendar end of the month in which the next decision falls.
    per = d.to_period("M") + (1 if chk_all.loc[d] else 0)
    return per.end_time.normalize()


STEPS = {d: min(len(pd.bdate_range(d + pd.offsets.BDay(1), next_check(d))), HORIZON_MAX) for d in FC_DATES}

TFM = walk_forward("timesfm", FC_DATES, CONTEXT, REFRESH_FORECASTS)
ARI = walk_forward("arima", FC_DATES, CONTEXT, REFRESH_FORECASTS)
print(f"{len(DECISIONS)} monthly decisions {DECISIONS[0].date()} -> {DECISIONS[-1].date()}"
      + (f" + live forecast from {LAST.date()}" if LIVE else ""))

356 monthly decisions 1997-02-28 -> 2026-09-30 + live forecast from 2026-10-19


In [5]:
def step_quantiles(store: dict) -> pd.DataFrame:
    # Forecast return deciles at step k (the next month-end) for every forecast date.
    rows = {d: store[d][STEPS[d] - 1] / series.loc[d] - 1 for d in FC_DATES}
    return pd.DataFrame.from_dict(rows, orient="index", columns=QCOLS)


def rw_quantiles() -> pd.DataFrame:
    rows = {}
    for d in FC_DATES:
        c = series.loc[:d].to_numpy(float)[-CONTEXT:]
        r = fc.random_walk_quantiles(c, STEPS[d], quantiles=DECILES, lookback=252)
        rows[d] = [r[k][-1] / c[-1] - 1 for k in QCOLS]
    return pd.DataFrame.from_dict(rows, orient="index", columns=QCOLS)


TQ, AQ, RQ = step_quantiles(TFM), step_quantiles(ARI), rw_quantiles()  # TimesFM, ARIMA, random walk

# Per-decision facts known at d (hurdle, momentum, trend) and the realised return to the next decision.
fdates = pd.DatetimeIndex(FC_DATES)
pos_chk = CHECK_DATES.get_indexer(fdates)  # -1 for the live (non month-end) date
nxt = [CHECK_DATES[i + 1] if 0 <= i < len(CHECK_DATES) - 1 else pd.NaT for i in pos_chk]
prv = [CHECK_DATES[i - 1] if i > 0 else CHECK_DATES[-1] for i in pos_chk]
yld = irx.reindex(irx.index.union(px_all.index)).ffill().reindex(fdates)
cal_days = np.array([(next_check(d) - d).days for d in fdates])
sma_on, sma = sma_signal(px_all["close"], SMA_WINDOW)

fq = pd.DataFrame(index=fdates)
fq["k"] = [STEPS[d] for d in fdates]
fq["next"] = nxt
fq["real"] = [series.loc[n] / series.loc[d] - 1 if pd.notna(n) else np.nan for d, n in zip(fdates, nxt)]
fq["cash_k"] = 0.0 if CASH_SOURCE == "zero" else (1 + yld.to_numpy() / 100) ** (cal_days / 365) - 1
fq["mom"] = [series.loc[d] / series.loc[p] - 1 for d, p in zip(fdates, prv)]
fq["sma_on"] = sma_on.reindex(fdates)


def prob_above(q: pd.DataFrame, x: pd.Series) -> pd.Series:
    # P(R > x) by linear interpolation of the forecast deciles (5% / 95% beyond q10 / q90).
    out = []
    for row, xi in zip(q.to_numpy(), x.to_numpy()):
        row = np.maximum.accumulate(row)
        out.append(1 - np.interp(xi, row, DECILES, left=0.05, right=0.95))
    return pd.Series(out, index=q.index)


fq["p_up"] = prob_above(TQ, fq["cash_k"])
EVAL = fq.index[fq["real"].notna()]  # decisions whose month has completed
print(f"{len(EVAL)} completed forecast months, {EVAL[0]:%Y-%m} -> {EVAL[-1]:%Y-%m}")
display(pd.concat([fq[["k", "real", "cash_k"]], TQ.add_prefix("tfm_")[["tfm_q10", "tfm_q50", "tfm_q90"]],
                   AQ[["q50"]].add_prefix("arima_"), fq[["p_up"]]], axis=1).tail(4).style.format("{:.2%}", subset=pd.IndexSlice[:, ["real", "cash_k", "tfm_q10", "tfm_q50", "tfm_q90", "arima_q50", "p_up"]], na_rep="-"))

355 completed forecast months, 1997-02 -> 2026-08


,k,real,cash_k,tfm_q10,tfm_q50,tfm_q90,arima_q50,p_up
2026-07-31 00:00:00,21,2.68%,0.31%,-2.80%,0.68%,3.44%,1.36%,55.07%
2026-08-31 00:00:00,22,-0.33%,0.30%,-2.37%,1.11%,3.75%,1.47%,62.58%
2026-09-30 00:00:00,22,-,0.34%,-2.11%,1.40%,4.11%,1.70%,68.89%
2026-10-19 00:00:00,9,-,0.13%,-1.64%,0.77%,2.75%,0.91%,65.25%


## 3. Forecast quality

Before any trading: do the forecasts carry information about the next month? Each score is compared with a naive benchmark, since a number on its own says little.

* **Direction**: share of months where the sign of the forecast median matches the realised return, against **always predicting "up"** (SPY rose in most months, so 50% is the wrong yardstick). Binomial test against the always-up hit rate.
* **Ranking (IC)**: Spearman correlation between forecast median and realised return. The t-stat comes from a regression of realised on forecast ranks with Newey-West errors.
* **Calibration**: a well-calibrated decile forecast puts 10% of realised returns in each of its 10 bins (PIT histogram), and the q10-q90 band covers 80%.
* **Accuracy**: mean pinball (quantile) loss over the 9 deciles; skill = 1 - loss / random-walk loss (> 0 = better than no information). Diebold-Mariano style t-stat on the loss difference.
* **Volatility**: does a wider forecast band precede bigger moves? Spearman(q90 - q10, |realised|).

In [6]:
real = fq.loc[EVAL, "real"]
MODELS = {"TimesFM": TQ, "ARIMA": AQ, "Random walk": RQ}


def nw_t(y, x) -> float:
    beta, se = newey_west(y, np.column_stack([np.ones(len(x)), x]), NW_LAGS)
    return beta[1] / se[1]


def direction_row(pred: pd.Series) -> dict:
    ok = pred.notna() & real.notna()
    p, r = pred[ok], real[ok]
    hits = int(((p > 0) == (r > 0)).sum())
    base = (r > 0).mean()
    return {
        "n": int(ok.sum()), "% forecast up": (p > 0).mean(), "Hit rate": hits / ok.sum(),
        "Always-up hit rate": base,
        "p (hit > always-up)": stats.binomtest(hits, int(ok.sum()), base, alternative="greater").pvalue,
        "IC (Spearman)": stats.spearmanr(p, r).statistic,
        "IC t-stat (NW)": nw_t(r.rank().to_numpy(), p.rank().to_numpy()),  # regression on ranks = Spearman
    }


dir_tab = pd.DataFrame({
    "TimesFM median": direction_row(TQ.loc[EVAL, "q50"]),
    "ARIMA median": direction_row(AQ.loc[EVAL, "q50"]),
    "1-month momentum": direction_row(fq.loc[EVAL, "mom"]),
}).T
print("Direction and ranking of next-month returns")
display(dir_tab.style.format({"n": "{:.0f}", "% forecast up": "{:.0%}", "Hit rate": "{:.1%}", "Always-up hit rate": "{:.1%}",
                              "p (hit > always-up)": "{:.3f}", "IC (Spearman)": "{:+.3f}", "IC t-stat (NW)": "{:+.2f}"}))

fig = make_subplots(rows=1, cols=2, subplot_titles=("TimesFM", "ARIMA"))
for col, (name, q, color) in enumerate([("TimesFM", TQ, C_STRAT), ("ARIMA", AQ, C_ARIMA)], start=1):
    x = q.loc[EVAL, "q50"]
    fig.add_trace(go.Scatter(x=x, y=real, mode="markers", marker=dict(color=color, size=5, opacity=0.6), name=name,
                             text=[d.strftime("%Y-%m") for d in EVAL], hovertemplate="%{text}<br>forecast %{x:.2%}<br>realised %{y:.2%}"),
                  row=1, col=col)
    fig.add_hline(y=0, line_color="grey", line_width=1, row=1, col=col)
    fig.add_vline(x=0, line_color="grey", line_width=1, row=1, col=col)
fig.update_xaxes(tickformat=".1%", title_text="forecast median return")
fig.update_yaxes(tickformat=".0%")
fig.update_yaxes(title_text="realised return", row=1, col=1)
fig.update_layout(height=420, showlegend=False, title="Forecast median vs realised monthly return")
fig.show()

Direction and ranking of next-month returns


,n,% forecast up,Hit rate,Always-up hit rate,p (hit > always-up),IC (Spearman),IC t-stat (NW)
TimesFM median,355,86%,60.6%,63.7%,0.897,+0.048,+0.94
ARIMA median,355,82%,59.4%,63.7%,0.956,+0.024,+0.46
1-month momentum,355,64%,56.3%,63.7%,0.998,-0.027,-0.51


In [7]:
def pit_bins(q: pd.DataFrame) -> np.ndarray:
    # Bin 0 = below q10, ..., bin 9 = above q90.
    qq = np.maximum.accumulate(q.loc[EVAL].to_numpy(), axis=1)
    return np.array([np.searchsorted(row, y) for row, y in zip(qq, real.to_numpy())])


def pinball(q: pd.DataFrame) -> pd.Series:
    u = real.to_numpy()[:, None] - q.loc[EVAL].to_numpy()
    return pd.Series(np.maximum(DECILES * u, (DECILES - 1) * u).mean(axis=1), index=EVAL)


loss_rw = pinball(RQ)
cal_rows, pit = {}, {}
for name, q in MODELS.items():
    ok = q.loc[EVAL].notna().all(axis=1)
    b = pit_bins(q)
    counts = np.bincount(b[ok.to_numpy()], minlength=10)
    pit[name] = counts / counts.sum()
    qe = q.loc[EVAL]
    loss = pinball(q)
    d = (loss - loss_rw)[ok]
    dm = d.mean() / newey_west(d.to_numpy(), np.ones((len(d), 1)), NW_LAGS)[1][0] if name != "Random walk" else np.nan
    width = (q.loc[EVAL, "q90"] - q.loc[EVAL, "q10"])[ok]
    cal_rows[name] = {
        "Coverage q10-q90 (target 80%)": ((real >= qe["q10"]) & (real <= qe["q90"]))[ok].mean(),
        "Coverage q30-q70 (target 40%)": ((real >= qe["q30"]) & (real <= qe["q70"]))[ok].mean(),
        "Below q10 (target 10%)": (real < qe["q10"])[ok].mean(),
        "Above q90 (target 10%)": (real > qe["q90"])[ok].mean(),
        "PIT uniformity p (chi2)": stats.chisquare(counts).pvalue,
        "Mean pinball loss": loss[ok].mean(),
        "Skill vs random walk": 1 - loss[ok].mean() / loss_rw[ok].mean(),
        "DM t-stat (loss - RW)": dm,
        "Width vs |realised| (Spearman)": stats.spearmanr(width, real[ok].abs()).statistic,
        "Mean q10-q90 width": width.mean(),
    }
cal_tab = pd.DataFrame(cal_rows).T
PCTC = ["Coverage q10-q90 (target 80%)", "Coverage q30-q70 (target 40%)", "Below q10 (target 10%)",
        "Above q90 (target 10%)", "Skill vs random walk", "Mean q10-q90 width"]
print("Calibration, accuracy and volatility skill (DM t < -2: significantly lower loss than the random walk)")
display(cal_tab.style.format({**{c: "{:.1%}" for c in PCTC}, "PIT uniformity p (chi2)": "{:.3f}",
                              "Mean pinball loss": "{:.5f}", "DM t-stat (loss - RW)": "{:+.2f}",
                              "Width vs |realised| (Spearman)": "{:+.2f}"}))

labels = ["<q10"] + [f"q{10 * i}-q{10 * (i + 1)}" for i in range(1, 9)] + [">q90"]
fig = go.Figure([go.Bar(x=labels, y=v, name=n, marker_color=c) for (n, v), c in zip(pit.items(), [C_STRAT, C_ARIMA, C_CASH])])
fig.add_hline(y=0.1, line_dash="dash", line_color="black", annotation_text="perfect calibration (10%)")
fig.update_layout(title="Where realised monthly returns fell inside the forecast deciles (PIT histogram)",
                  yaxis_tickformat=".0%", barmode="group", height=400)
fig.show()

Calibration, accuracy and volatility skill (DM t < -2: significantly lower loss than the random walk)


,Coverage q10-q90 (target 80%),Coverage q30-q70 (target 40%),Below q10 (target 10%),Above q90 (target 10%),PIT uniformity p (chi2),Mean pinball loss,Skill vs random walk,DM t-stat (loss - RW),Width vs |realised| (Spearman),Mean q10-q90 width
TimesFM,74.4%,30.1%,11.8%,13.8%,0.001,0.01312,4.6%,-2.66,+0.40,10.4%
ARIMA,81.7%,45.9%,11.0%,7.3%,0.004,0.01364,0.9%,-0.54,+0.06,12.0%
Random walk,84.5%,45.6%,7.0%,8.5%,0.000,0.01376,0.0%,+nan,+0.27,13.5%


In [8]:
# Stability: rolling 60-month rank correlation and hit rate (TimesFM vs ARIMA vs always-up)
ROLL = 60


def rolling_spearman(p: pd.Series, r: pd.Series) -> pd.Series:
    vals = [np.nan] * (ROLL - 1) + [stats.spearmanr(p.iloc[i - ROLL : i], r.iloc[i - ROLL : i]).statistic
                                    for i in range(ROLL, len(p) + 1)]
    return pd.Series(vals, index=p.index)


roll = pd.DataFrame(index=EVAL)
for name, q in (("TimesFM", TQ), ("ARIMA", AQ)):
    p = q.loc[EVAL, "q50"]
    roll[f"{name} IC"] = rolling_spearman(p, real)
    roll[f"{name} hit rate"] = ((p > 0) == (real > 0)).astype(float).rolling(ROLL).mean()
roll["Always-up hit rate"] = (real > 0).astype(float).rolling(ROLL).mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=(f"Rolling {ROLL}-month rank correlation (forecast median vs realised)",
                                    f"Rolling {ROLL}-month direction hit rate"))
for name, color in (("TimesFM", C_STRAT), ("ARIMA", C_ARIMA)):
    fig.add_trace(go.Scatter(x=roll.index, y=roll[f"{name} IC"], name=f"{name} IC", line=dict(color=color)), row=1, col=1)
    fig.add_trace(go.Scatter(x=roll.index, y=roll[f"{name} hit rate"], name=f"{name} hit rate",
                             line=dict(color=color, dash="dot")), row=2, col=1)
fig.add_trace(go.Scatter(x=roll.index, y=roll["Always-up hit rate"], name="always-up", line=dict(color=C_BH)), row=2, col=1)
fig.add_hline(y=0, line_color="black", line_width=1, row=1, col=1)
fig.add_vline(x=pd.Timestamp(MODEL_RELEASE), line_dash="dash", line_color="red")
fig.update_yaxes(tickformat=".0%", row=2, col=1)
fig.update_layout(height=520, hovermode="x unified",
                  title=f"Forecast skill over time (red dashed = TimesFM 2.5 release, {MODEL_RELEASE})")
fig.show()

## 4. Trading rules and backtest

All signals are formed at the decision close $d$ from the forecast (and, for R4 / SMA200, the close vs its SMA), then run through the **same engine as the SMA backtest** (`bt_common.backtest_signal`): the signal is read only at month-ends, filled at the next open, and costs are charged per switch. The hurdle $c$ is the T-bill yield known at $d$, accrued over the calendar days to the next month-end.

In [9]:
cash = fq["cash_k"]
dec = pd.DatetimeIndex(DECISIONS)  # the live (non month-end) forecast never trades
RULES = {
    "R1": ("R1 TimesFM median", TQ["q50"] > cash),
    "R2": (f"R2 TimesFM P(R>c) >= {P_STAR:.0%}", fq["p_up"] >= P_STAR),
    "R3": (f"R3 TimesFM q10 >= {TAIL_Q10:.0%}", TQ["q10"] >= TAIL_Q10),
    "R4": (f"R4 TimesFM median & SMA{SMA_WINDOW}", (TQ["q50"] > cash) & (fq["sma_on"] == 1)),
    "SMA": (f"SMA{SMA_WINDOW} monthly", fq["sma_on"] == 1),
    "ARIMA": ("ARIMA median", (AQ["q50"] > cash).where(AQ["q50"].notna())),
    "MOM": ("1-month momentum", fq["mom"] > cash),
}
LABEL = {k: v[0] for k, v in RULES.items()}
SIG = {k: v[1].reindex(dec).astype(float) for k, v in RULES.items()}


def run(signal: pd.Series, cost_bps=COST_BPS, execution=EXECUTION) -> pd.DataFrame:
    return backtest_signal(px_all, signal, CASH, cost_bps, execution, start=START, rebalance=REBALANCE)


RES = {k: run(s) for k, s in SIG.items()}
res = RES[MAIN_RULE]
trades = trade_log(res, px_all, EXECUTION)
spans = {(r.index[0], r.index[-1]) for r in RES.values()}
assert len(spans) == 1, "all strategies must cover the same window"
print(f"Backtest {res.index[0].date()} -> {res.index[-1].date()} | rebalance {REBALANCE} | {EXECUTION} | "
      f"{COST_BPS} bp/switch | cash={CASH_SOURCE} | main rule {LABEL[MAIN_RULE]}")

Backtest 1997-04-01 -> 2026-10-19 | rebalance monthly | next_open | 5.0 bp/switch | cash=irx | main rule R1 TimesFM median


## 5. Results

In [10]:
summary = pd.DataFrame({LABEL[k]: {**perf_stats(r["r_strat"], r["r_cash"]), **strategy_stats(r)} for k, r in RES.items()})
summary[f"Buy & hold {EQUITY_TICKER}"] = {**perf_stats(res["r_bh"], res["r_cash"]), "Time in market": 1.0, "Switches": 0,
                                          "Switches / yr": 0.0, "Cost drag / yr": 0.0}
summary["T-bills"] = {**perf_stats(res["r_cash"], res["r_cash"]), "Time in market": 0.0, "Switches": 0,
                      "Switches / yr": 0.0, "Cost drag / yr": 0.0}
PCT_ROWS = {"CAGR", "Vol", "Max DD", "Worst day", "Time in market", "Cost drag / yr"}


def _fmt(row, v):
    if isinstance(v, (float, np.floating)) and not pd.isna(v):
        return f"{v:.2%}" if row in PCT_ROWS else f"{v:.2f}"
    return v


print("Net of costs; Sharpe / Sortino on returns in excess of T-bills")
display(summary.apply(lambda col: [_fmt(r, v) for r, v in col.items()]).set_axis(summary.index))

Net of costs; Sharpe / Sortino on returns in excess of T-bills


,R1 TimesFM median,R2 TimesFM P(R>c) >= 55%,R3 TimesFM q10 >= -8%,R4 TimesFM median & SMA200,SMA200 monthly,ARIMA median,1-month momentum,Buy & hold SPY,T-bills
CAGR,9.79%,8.02%,8.40%,8.56%,9.77%,7.26%,5.94%,9.96%,2.18%
Vol,17.28%,14.55%,15.67%,12.15%,12.76%,16.83%,12.23%,19.17%,0.17%
Sharpe,0.50,0.46,0.46,0.56,0.63,0.37,0.36,0.48,NaN
Sortino,0.70,0.63,0.63,0.78,0.87,0.52,0.50,0.67,NaN
Max DD,-44.10%,-32.05%,-42.27%,-20.66%,-25.41%,-54.12%,-31.30%,-55.42%,-0.00%
Calmar,0.22,0.25,0.20,0.41,0.38,0.13,0.19,0.18,NaN
Worst day,-12.94%,-9.29%,-9.29%,-9.29%,-9.29%,-12.94%,-9.29%,-12.94%,-0.00%
MaxDD peak,2000-09-01,2020-02-19,2000-09-01,2022-01-04,2022-01-04,2000-09-01,2021-09-02,2007-10-11,2020-03-20
MaxDD recovered,2009-12-28,2020-08-10,2007-04-20,2024-03-08,2024-06-18,2016-12-08,2025-06-30,2012-08-17,2020-04-13
Final x,15.78,9.76,10.84,11.31,15.73,7.92,5.50,16.53,1.89


In [11]:
sma_res = RES["SMA"]
curves = pd.DataFrame(
    {
        f"{LABEL[MAIN_RULE]} net": (1 + res["r_strat"]).cumprod(),
        f"Buy & hold {EQUITY_TICKER}": (1 + res["r_bh"]).cumprod(),
        f"SMA{SMA_WINDOW} monthly net": (1 + sma_res["r_strat"]).cumprod(),
        "T-bills": (1 + res["r_cash"]).cumprod(),
    }
)
dd = curves / curves.cummax() - 1
w_eq = res["pos"]
COLORS = [C_STRAT, C_BH, C_SMA, C_CASH]

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, row_heights=[0.55, 0.17, 0.28], vertical_spacing=0.05,
    subplot_titles=(f"Growth of $1 (log scale), {res.index[0].date()} -> {res.index[-1].date()}",
                    f"{LABEL[MAIN_RULE]}: weights", "Drawdown"),
)
for name, color in zip(curves.columns, COLORS):
    fig.add_trace(go.Scatter(x=curves.index, y=curves[name], name=name, line=dict(color=color, width=1.4),
                             legendgroup="growth"), row=1, col=1)
fig.add_trace(go.Scatter(x=w_eq.index, y=w_eq, name=f"{EQUITY_TICKER} weight", line_shape="hv",
                         line=dict(color=C_BH, width=0), fill="tozeroy", fillcolor="rgba(255,127,14,0.45)",
                         legendgroup="w"), row=2, col=1)
fig.add_trace(go.Scatter(x=w_eq.index, y=np.ones(len(w_eq)), name="T-bill weight", line_shape="hv",
                         line=dict(color=C_CASH, width=0), fill="tonexty", fillcolor="rgba(44,160,44,0.4)",
                         legendgroup="w", hoverinfo="skip"), row=2, col=1)
fig.add_trace(go.Scatter(x=w_eq.index, y=w_eq.rolling(252).mean(), name=f"{EQUITY_TICKER} weight, 1y avg",
                         line=dict(color="black", width=1), legendgroup="w"), row=2, col=1)
for name, color in zip(curves.columns[:3], COLORS):
    fig.add_trace(go.Scatter(x=dd.index, y=dd[name], name=f"DD {name}", line=dict(color=color, width=1),
                             showlegend=False), row=3, col=1)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(tickformat=".0%", range=[0, 1.02], row=2, col=1)
fig.update_yaxes(tickformat=".0%", row=3, col=1)
fig.update_layout(height=820, hovermode="x unified", legend=dict(orientation="h", y=-0.08))
fig.show()

In [12]:
# Last 5 years: each month's forecast range (q10-q90 bar, q50 dot) plotted at the month it targets
YEARS = 5
t0 = LAST - pd.DateOffset(years=YEARS)
show = [d for d in FC_DATES if d >= t0]
target_x = [fq.at[d, "next"] if pd.notna(fq.at[d, "next"]) else next_check(d) for d in show]
base = series.loc[show].to_numpy()
q50 = base * (1 + TQ.loc[show, "q50"].to_numpy())
on = (TQ.loc[show, "q50"] > fq.loc[show, "cash_k"]).to_numpy()
recent = series.loc[t0:]

fig = go.Figure()
fig.add_trace(go.Scatter(x=recent.index, y=recent, name=f"{EQUITY_TICKER} ({INPUT_FIELD})", line=dict(color=C_BH, width=1.2)))
for mask, name, color in ((on, "forecast: risk-on (R1)", "green"), (~on, "forecast: risk-off (R1)", "red")):
    fig.add_trace(go.Scatter(
        x=np.array(target_x)[mask], y=q50[mask], mode="markers", name=name, marker=dict(color=color, size=7),
        error_y=dict(type="data", symmetric=False, color=color, thickness=1,
                     array=(base * (1 + TQ.loc[show, "q90"].to_numpy()) - q50)[mask],
                     arrayminus=(q50 - base * (1 + TQ.loc[show, "q10"].to_numpy()))[mask]),
    ))
off = (res["pos"].loc[t0:] == 0).astype(int)
grp = (off != off.shift()).cumsum()
for _, g in off[off == 1].groupby(grp[off == 1]):
    fig.add_vrect(x0=g.index[0], x1=g.index[-1], fillcolor="red", opacity=0.08, line_width=0, layer="below")
fig.update_layout(title=f"Last {YEARS} years: TimesFM one-month-ahead range (q10-q90) vs realised; "
                        f"shaded = {LABEL[MAIN_RULE]} in T-bills", height=520, hovermode="x unified",
                  legend=dict(orientation="h", y=-0.15))
fig.show()

In [13]:
yearly = pd.DataFrame(
    {
        f"{MAIN_RULE} net": (1 + res["r_strat"]).groupby(res.index.year).prod() - 1,
        f"SMA{SMA_WINDOW} net": (1 + sma_res["r_strat"]).groupby(sma_res.index.year).prod() - 1,
        "Buy & hold": (1 + res["r_bh"]).groupby(res.index.year).prod() - 1,
        "T-bills": (1 + res["r_cash"]).groupby(res.index.year).prod() - 1,
    }
)
yearly[f"{MAIN_RULE} - B&H"] = yearly.iloc[:, 0] - yearly["Buy & hold"]
yearly[f"{MAIN_RULE} time in market"] = res["pos"].groupby(res.index.year).mean()
yearly["Switches"] = (res["turnover"] > 0).groupby(res.index.year).sum()

fig = go.Figure()
for name, color in zip(yearly.columns[:3], [C_STRAT, C_SMA, C_BH]):
    fig.add_trace(go.Bar(x=yearly.index, y=yearly[name], name=name, marker_color=color))
fig.update_layout(title="Calendar-year returns", barmode="group", yaxis_tickformat=".0%", height=420)
fig.show()
display(yearly.style.format({c: "{:.1%}" for c in yearly.columns if c != "Switches"}))

,R1 net,SMA200 net,Buy & hold,T-bills,R1 - B&H,R1 time in market,Switches
Date,,,,,,,
1997,27.4%,27.4%,27.4%,3.8%,-0.1%,100.0%,1
1998,28.9%,12.1%,28.9%,4.8%,0.0%,100.0%,0
1999,20.4%,13.2%,20.4%,4.6%,0.0%,100.0%,0
2000,-7.7%,0.3%,-7.7%,5.8%,0.0%,100.0%,0
2001,-7.4%,3.4%,-12.3%,3.4%,4.8%,41.9%,3
2002,-7.9%,1.6%,-23.0%,1.6%,15.1%,32.5%,3
2003,7.4%,22.9%,28.5%,1.0%,-21.1%,33.7%,5
2004,4.0%,7.7%,11.2%,1.4%,-7.1%,65.9%,4
2005,0.0%,1.8%,4.7%,3.1%,-4.7%,7.9%,2


In [14]:
# Rolling 3-year annualised return: strategy minus buy & hold
WIN = 252 * 3


def roll_ann(r):
    return np.exp(np.log1p(r).rolling(WIN).sum() * 252 / WIN) - 1


fig = go.Figure()
for k, color in ((MAIN_RULE, C_STRAT), ("SMA", C_SMA)):
    rel = (roll_ann(RES[k]["r_strat"]) - roll_ann(res["r_bh"])).dropna()
    fig.add_trace(go.Scatter(x=rel.index, y=rel, name=LABEL[k], line=dict(color=color, width=1)))
    print(f"{LABEL[k]} beat buy & hold in {(rel > 0).mean():.0%} of rolling 3-year windows")
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(title="Rolling 3-year annualised return minus buy & hold", yaxis_tickformat=".0%",
                  height=380, hovermode="x unified")
fig.show()

R1 TimesFM median beat buy & hold in 41% of rolling 3-year windows
SMA200 monthly beat buy & hold in 31% of rolling 3-year windows


## 6. Trades and current signal

In [15]:
print(f"{LABEL[MAIN_RULE]}: last trades")
display(trades.assign(filled=trades["filled"].dt.date).tail(8))

d_last = DECISIONS[-1]
now = pd.DataFrame({
    "as of close": [d_last.date()] + ([LAST.date()] if LIVE else []),
    "kind": ["last month-end decision"] + (["live (not a decision date)"] if LIVE else []),
    "target month-end": [next_check(d_last).date()] + ([next_check(LAST).date()] if LIVE else []),
})
rows = [d_last] + ([LAST] if LIVE else [])
now["TimesFM q10"] = TQ.loc[rows, "q10"].to_numpy()
now["TimesFM q50"] = TQ.loc[rows, "q50"].to_numpy()
now["TimesFM q90"] = TQ.loc[rows, "q90"].to_numpy()
now["T-bill hurdle"] = fq.loc[rows, "cash_k"].to_numpy()
now["P(R > hurdle)"] = fq.loc[rows, "p_up"].to_numpy()
now["ARIMA q50"] = AQ.loc[rows, "q50"].to_numpy()
now[f"close > SMA{SMA_WINDOW}"] = fq.loc[rows, "sma_on"].map({1.0: "yes", 0.0: "no"}).to_numpy()
PCT_COLS = ["TimesFM q10", "TimesFM q50", "TimesFM q90", "T-bill hurdle", "ARIMA q50"]
display(now.set_index("as of close").style.format({**{c: "{:+.2%}" for c in PCT_COLS}, "P(R > hurdle)": "{:.0%}"}))

state = {LABEL[k]: {1.0: "equity", 0.0: "T-bills"}.get(SIG[k].loc[d_last], "unchanged (no forecast)") for k in SIG}
print(f"Positions decided at the {d_last.date()} close (held until the next month-end decision):")
for name, s in state.items():
    print(f"  {name:<32} -> {s}")
if LIVE:
    print(f"Live forecast from {LAST.date()} is informational only: the rule acts at the month-end close "
          f"(~{next_check(LAST).date()}, holidays ignored).")
print("Research output only -- not a recommendation or an order.")

R1 TimesFM median: last trades


,filled,action,fill price
39,2019-05-01,SELL -> T-bills,294.72
40,2019-06-03,BUY equity,275.31
41,2020-09-01,SELL -> T-bills,350.21
42,2020-10-01,BUY equity,337.69
43,2022-12-01,SELL -> T-bills,408.77
44,2023-01-03,BUY equity,384.37
45,2023-11-01,SELL -> T-bills,419.20
46,2023-12-01,BUY equity,455.77


,kind,target month-end,TimesFM q10,TimesFM q50,TimesFM q90,T-bill hurdle,P(R > hurdle),ARIMA q50,close > SMA200
as of close,,,,,,,,,
2026-09-30,last month-end decision,2026-10-31,-2.11%,+1.40%,+4.11%,+0.34%,69%,+1.70%,yes
2026-10-19,live (not a decision date),2026-10-31,-1.64%,+0.77%,+2.75%,+0.13%,65%,+0.91%,yes


Positions decided at the 2026-09-30 close (held until the next month-end decision):
  R1 TimesFM median                -> equity
  R2 TimesFM P(R>c) >= 55%         -> equity
  R3 TimesFM q10 >= -8%            -> equity
  R4 TimesFM median & SMA200       -> equity
  SMA200 monthly                   -> equity
  ARIMA median                     -> equity
  1-month momentum                 -> T-bills
Live forecast from 2026-10-19 is informational only: the rule acts at the month-end close (~2026-10-31, holidays ignored).
Research output only -- not a recommendation or an order.


## 7. Robustness

* **Is the edge real?** Mean next-month excess return when each rule says risk-on vs risk-off, with Newey-West and block-bootstrap inference (as in `sma200_regime_analysis.ipynb`).
* **Random-timing placebo**: shuffle each rule's monthly positions (same time in market) and see where the real Sharpe ratio ranks.
* **Thresholds and costs**: R2 / R3 thresholds swept, costs 0 / 5 / 20 bp, execution timing. Picking the best threshold in-sample is overfitting -- the sweep is descriptive only.
* **Sub-periods**, including the months after TimesFM 2.5 was published: the only truly out-of-sample window, because the model may have seen S&P 500 history in pre-training.

In [16]:
LAG = EXECUTIONS[EXECUTION][1]
PANELS = {k: decision_panel(r, LAG) for k, r in RES.items()}


def on_off_inference(pan: pd.DataFrame) -> dict:
    ex = pan["excess"].to_numpy()
    on = (pan["state"] == 1).to_numpy()
    if on.all() or (~on).all():
        return {"n on": int(on.sum()), "n off": int((~on).sum())}
    idx = block_indices(len(pan), N_BOOT, BLOCK, RNG)
    ex_b, on_b = ex[idx], on[idx]
    with np.errstate(invalid="ignore", divide="ignore"):
        diff_b = (ex_b * on_b).sum(1) / on_b.sum(1) - (ex_b * ~on_b).sum(1) / (~on_b).sum(1)
    beta, se = newey_west(ex, np.column_stack([np.ones_like(ex), on]).astype(float), NW_LAGS)
    return {
        "n on": int(on.sum()), "n off": int((~on).sum()),
        "Mean excess, on": ex[on].mean(), "Mean excess, off": ex[~on].mean(),
        "Difference (on - off)": beta[1], "t-stat (NW)": beta[1] / se[1],
        "95% CI (bootstrap)": f"[{np.nanquantile(diff_b, 0.025):+.2%}, {np.nanquantile(diff_b, 0.975):+.2%}]",
        "p (on > off, bootstrap)": np.nanmean(diff_b <= 0),
    }


inf_tab = pd.DataFrame({LABEL[k]: on_off_inference(p) for k, p in PANELS.items()}).T
print("Monthly excess return of SPY over T-bills in the month after each decision, by the rule's call")
display(inf_tab.style.format({"Mean excess, on": "{:+.2%}", "Mean excess, off": "{:+.2%}", "Difference (on - off)": "{:+.2%}",
                              "t-stat (NW)": "{:+.2f}", "p (on > off, bootstrap)": "{:.3f}"}, na_rep="-"))

Monthly excess return of SPY over T-bills in the month after each decision, by the rule's call


,n on,n off,"Mean excess, on","Mean excess, off",Difference (on - off),t-stat (NW),95% CI (bootstrap),"p (on > off, bootstrap)"
R1 TimesFM median,284,68,+0.83%,+0.13%,+0.71%,+1.12,"[-0.49%, +1.99%]",0.134
R2 TimesFM P(R>c) >= 55%,226,126,+0.83%,+0.46%,+0.37%,+0.65,"[-0.73%, +1.51%]",0.272
R3 TimesFM q10 >= -8%,306,46,+0.63%,+1.13%,-0.50%,-0.53,"[-2.46%, +1.54%]",0.728
R4 TimesFM median & SMA200,221,131,+0.85%,+0.44%,+0.41%,+0.73,"[-0.68%, +1.49%]",0.252
SMA200 monthly,261,91,+0.85%,+0.26%,+0.59%,+0.80,"[-0.97%, +1.99%]",0.241
ARIMA median,280,72,+0.59%,+1.11%,-0.52%,-0.94,"[-1.64%, +0.57%]",0.827
1-month momentum,219,133,+0.59%,+0.88%,-0.29%,-0.58,"[-1.31%, +0.65%]",0.739


In [17]:
# Random-timing placebo on the monthly holding periods: same number of risk-on months, random order.
pan0 = PANELS[MAIN_RULE]
r_eq, r_tb = pan0["r_eq"].to_numpy(), pan0["r_tbill"].to_numpy()


def monthly_sharpe(pos: np.ndarray) -> np.ndarray:
    # pos: (n_draws, n_months) 0/1; costs charged on every change (and the initial entry).
    pos = np.atleast_2d(pos)
    turn = np.abs(np.diff(pos, axis=1, prepend=0))
    ex = pos * r_eq + (1 - pos) * r_tb - turn * COST_BPS / 1e4 - r_tb
    return ex.mean(1) / ex.std(1, ddof=1) * math.sqrt(12)


rows, draws = {}, {}
for k, pan in PANELS.items():
    s = pan["state"].reindex(pan0.index).to_numpy()
    if np.isnan(s).any() or s.min() == s.max():
        continue
    actual = monthly_sharpe(s)[0]
    perm = RNG.permuted(np.tile(s, (N_PLACEBO, 1)), axis=1)
    draws[k] = monthly_sharpe(perm)
    rows[LABEL[k]] = {"time in market": s.mean(), "Sharpe (monthly basis)": actual,
                      "placebo median": np.median(draws[k]), "placebo 95th pct": np.quantile(draws[k], 0.95),
                      "share of placebos beaten": (draws[k] < actual).mean()}
rows[f"Buy & hold {EQUITY_TICKER}"] = {"time in market": 1.0, "Sharpe (monthly basis)": monthly_sharpe(np.ones(len(pan0)))[0]}
placebo_tab = pd.DataFrame(rows).T
display(placebo_tab.style.format({"time in market": "{:.0%}", "Sharpe (monthly basis)": "{:.2f}", "placebo median": "{:.2f}",
                                  "placebo 95th pct": "{:.2f}", "share of placebos beaten": "{:.0%}"}, na_rep="-"))

fig = go.Figure()
for k, color in ((MAIN_RULE, C_STRAT), ("SMA", C_SMA)):
    if k in draws:
        fig.add_trace(go.Histogram(x=draws[k], name=f"placebo, {LABEL[k]}", marker_color=color, opacity=0.45, nbinsx=60))
        fig.add_vline(x=placebo_tab.loc[LABEL[k], "Sharpe (monthly basis)"], line_color=color, line_width=2,
                      annotation_text=LABEL[k])
fig.update_layout(barmode="overlay", height=380, xaxis_title="Sharpe ratio (monthly, annualised)",
                  title=f"Random-timing placebo ({N_PLACEBO:,} shuffles of the same monthly positions)")
fig.show()

,time in market,Sharpe (monthly basis),placebo median,placebo 95th pct,share of placebos beaten
R1 TimesFM median,81%,0.55,0.45,0.60,86%
R2 TimesFM P(R>c) >= 55%,64%,0.50,0.40,0.59,82%
R3 TimesFM q10 >= -8%,87%,0.50,0.49,0.60,56%
R4 TimesFM median & SMA200,63%,0.63,0.40,0.60,97%
SMA200 monthly,74%,0.71,0.43,0.61,100%
ARIMA median,80%,0.39,0.45,0.60,25%
1-month momentum,62%,0.39,0.39,0.60,49%
Buy & hold SPY,100%,0.53,-,-,-


In [18]:
FMT = {"CAGR": "{:.2%}", "Sharpe": "{:.2f}", "Max DD": "{:.1%}", "Switches / yr": "{:.1f}", "Time in market": "{:.0%}"}


def row(b):
    s, st = perf_stats(b["r_strat"], b["r_cash"]), strategy_stats(b)
    return {"CAGR": s["CAGR"], "Sharpe": s["Sharpe"], "Max DD": s["Max DD"],
            "Switches / yr": st["Switches / yr"], "Time in market": st["Time in market"]}


bh_s = perf_stats(res["r_bh"], res["r_cash"])
sweeps = {
    "R2: P(R > c) threshold": ("p", [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70], P_STAR,
                               lambda p: (fq["p_up"] >= p).reindex(dec).astype(float)),
    "R3: q10 threshold": ("x", [-0.03, -0.04, -0.05, -0.06, -0.07, -0.08, -0.10, -0.12, -0.15], TAIL_Q10,
                          lambda x: (TQ["q10"] >= x).reindex(dec).astype(float)),
}
fig = make_subplots(rows=2, cols=3, subplot_titles=("CAGR", "Sharpe", "Max DD") + ("",) * 3, vertical_spacing=0.18)
for i, (title, (_, grid, chosen, make)) in enumerate(sweeps.items(), start=1):
    sw = pd.DataFrame({g: row(run(make(g))) for g in grid}).T
    for j, metric in enumerate(["CAGR", "Sharpe", "Max DD"], start=1):
        fig.add_trace(go.Scatter(x=sw.index, y=sw[metric], mode="lines+markers", name=title, showlegend=j == 1,
                                 line=dict(color=C_STRAT if i == 1 else C_ARIMA)), row=i, col=j)
        fig.add_hline(y=bh_s[metric], line_dash="dash", line_color="black", row=i, col=j)
        fig.add_vline(x=chosen, line_dash="dot", line_color="grey", row=i, col=j)
        if metric != "Sharpe":
            fig.update_yaxes(tickformat=".0%", row=i, col=j)
    fig.update_xaxes(title_text=title, tickformat=".0%", row=i, col=2)
fig.update_layout(title=f"Threshold sweeps ({COST_BPS} bp/switch; dashed = buy & hold, dotted = chosen value)", height=620)
fig.show()

cost_tab = pd.DataFrame({LABEL[k]: {f"Sharpe @ {c:.0f} bp": row(run(SIG[k], cost_bps=c))["Sharpe"] for c in (0.0, 5.0, 20.0)}
                         for k in SIG}).T
cost_tab["Switches / yr"] = [strategy_stats(RES[k])["Switches / yr"] for k in SIG]
print(f"Sharpe by cost per switch (buy & hold {bh_s['Sharpe']:.2f})")
display(cost_tab.style.format("{:.2f}"))

exe = pd.DataFrame({ex: row(run(SIG[MAIN_RULE], execution=ex)) for ex in EXECUTIONS}).T
print(f"{LABEL[MAIN_RULE]} by execution timing (same_close is look-ahead biased)")
display(exe.style.format(FMT))

Sharpe by cost per switch (buy & hold 0.48)


,Sharpe @ 0 bp,Sharpe @ 5 bp,Sharpe @ 20 bp,Switches / yr
R1 TimesFM median,0.51,0.50,0.49,1.59
R2 TimesFM P(R>c) >= 55%,0.46,0.46,0.43,2.47
R3 TimesFM q10 >= -8%,0.46,0.46,0.45,1.12
R4 TimesFM median & SMA200,0.57,0.56,0.54,1.86
SMA200 monthly,0.63,0.63,0.61,1.32
ARIMA median,0.38,0.37,0.36,0.98
1-month momentum,0.38,0.36,0.29,5.41


R1 TimesFM median by execution timing (same_close is look-ahead biased)


,CAGR,Sharpe,Max DD,Switches / yr,Time in market
next_open,9.79%,0.50,-44.1%,1.6,81%
next_close,10.06%,0.51,-46.0%,1.6,81%
same_close,9.97%,0.51,-45.7%,1.6,81%


In [19]:
PERIODS = {
    "1997-1999 bull": ("1997-04-01", "1999-12-31"),
    "2000-2002 dot-com bust": ("2000-03-24", "2002-10-09"),
    "2002-2007 recovery": ("2002-10-10", "2007-10-09"),
    "2007-2009 GFC": ("2007-10-10", "2009-03-09"),
    "2009-2019 bull": ("2009-03-10", "2019-12-31"),
    "2020 COVID crash": ("2020-02-19", "2020-03-23"),
    "2020-2021 rebound": ("2020-03-24", "2021-12-31"),
    "2022 bear": ("2022-01-03", "2022-10-12"),
    "2023-latest": ("2023-01-01", None),
    f"After TimesFM 2.5 release ({MODEL_RELEASE})": (MODEL_RELEASE, None),
}
recs = {}
for name, (a, b) in PERIODS.items():
    sl = {k: r.loc[a:b] for k, r in RES.items()}
    if len(sl[MAIN_RULE]) < 5:
        continue
    recs[name] = {**{LABEL[k]: (1 + s["r_strat"]).prod() - 1 for k, s in sl.items() if k in (MAIN_RULE, "R4", "SMA", "ARIMA")},
                  "Buy & hold": (1 + sl[MAIN_RULE]["r_bh"]).prod() - 1,
                  f"{MAIN_RULE} time in market": sl[MAIN_RULE]["pos"].mean()}
display(pd.DataFrame(recs).T.style.format("{:+.1%}"))

# Forecast skill before vs after the model was published (the "after" window is short: treat it as anecdotal).
split = {"Before release": EVAL[EVAL < pd.Timestamp(MODEL_RELEASE)], "After release": EVAL[EVAL >= pd.Timestamp(MODEL_RELEASE)]}
srows = {}
for name, idx in split.items():
    if len(idx) < 3:
        continue
    p, r = TQ.loc[idx, "q50"], fq.loc[idx, "real"]
    lt, lr = pinball(TQ).loc[idx], loss_rw.loc[idx]
    srows[name] = {"months": len(idx), "TimesFM hit rate": ((p > 0) == (r > 0)).mean(), "Always-up hit rate": (r > 0).mean(),
                   "IC (Spearman)": stats.spearmanr(p, r).statistic if len(idx) > 3 else np.nan,
                   "q10-q90 coverage": ((r >= TQ.loc[idx, "q10"]) & (r <= TQ.loc[idx, "q90"])).mean(),
                   "Skill vs random walk": 1 - lt.mean() / lr.mean()}
print("TimesFM forecast quality before / after its public release")
display(pd.DataFrame(srows).T.style.format({"months": "{:.0f}", "TimesFM hit rate": "{:.0%}", "Always-up hit rate": "{:.0%}",
                                           "IC (Spearman)": "{:+.2f}", "q10-q90 coverage": "{:.0%}",
                                           "Skill vs random walk": "{:+.1%}"}))

,R1 TimesFM median,R4 TimesFM median & SMA200,SMA200 monthly,ARIMA median,Buy & hold,R1 time in market
1997-1999 bull,+97.7%,+61.5%,+61.5%,+97.7%,+97.8%,+100.0%
2000-2002 dot-com bust,-22.9%,+3.2%,+3.2%,-33.1%,-45.3%,+53.4%
2002-2007 recovery,+30.9%,+34.2%,+73.9%,+32.0%,+115.0%,+45.0%
2007-2009 GFC,-39.7%,-4.1%,-4.1%,-43.6%,-55.0%,+76.9%
2009-2019 bull,+383.9%,+149.1%,+193.5%,+229.3%,+489.5%,+89.3%
2020 COVID crash,-31.8%,-11.4%,-11.4%,-31.8%,-31.8%,+100.0%
2020-2021 rebound,+120.7%,+65.9%,+60.7%,+113.9%,+113.9%,+95.3%
2022 bear,-23.8%,-16.2%,-16.2%,-23.8%,-23.8%,+100.0%
2023-latest,+93.1%,+59.3%,+60.7%,+92.7%,+111.3%,+97.7%
After TimesFM 2.5 release (2025-09-02),+20.1%,+9.1%,+9.1%,+20.1%,+20.1%,+100.0%


TimesFM forecast quality before / after its public release


,months,TimesFM hit rate,Always-up hit rate,IC (Spearman),q10-q90 coverage,Skill vs random walk
Before release,343,60%,64%,+0.06,74%,+4.7%
After release,12,67%,67%,-0.43,75%,+2.9%


In [20]:
# Context length: does TimesFM do better with a shorter or longer history? (RUN_CONTEXT_SWEEP; cached once computed)
if RUN_CONTEXT_SWEEP:
    ctx_rows = {}
    for c in (256, 512, CONTEXT):
        store = TFM if c == CONTEXT else walk_forward("timesfm", FC_DATES, c, REFRESH_FORECASTS)
        q = step_quantiles(store)
        p = q.loc[EVAL, "q50"]
        sig = (q["q50"] > cash).reindex(dec).astype(float)
        ctx_rows[c] = {"hit rate": ((p > 0) == (real > 0)).mean(), "IC (Spearman)": stats.spearmanr(p, real).statistic,
                       "skill vs RW": 1 - pinball(q).mean() / loss_rw.mean(), **row(run(sig))}
    display(pd.DataFrame(ctx_rows).T.rename_axis("context").style.format(
        {"hit rate": "{:.1%}", "IC (Spearman)": "{:+.3f}", "skill vs RW": "{:+.1%}", **FMT}))
else:
    print("Context sweep skipped (set RUN_CONTEXT_SWEEP = True; first run costs ~2 x 350 extra TimesFM forecasts).")

Context sweep skipped (set RUN_CONTEXT_SWEEP = True; first run costs ~2 x 350 extra TimesFM forecasts).


## 8. Evidence summary (computed)

In [21]:
t = dir_tab.loc["TimesFM median"]
c = cal_tab.loc["TimesFM"]
s_main, s_bh, s_sma = summary[LABEL[MAIN_RULE]], summary[f"Buy & hold {EQUITY_TICKER}"], summary[LABEL["SMA"]]
inf = inf_tab.loc[LABEL[MAIN_RULE]]
lines = [
    f"Direction: TimesFM median hit rate {t['Hit rate']:.1%} vs {t['Always-up hit rate']:.1%} for always-up "
    f"(p = {t['p (hit > always-up)']:.2f}); it forecasts 'up' in {t['% forecast up']:.0%} of months.",
    f"Ranking: Spearman IC {t['IC (Spearman)']:+.3f} (Newey-West t {t['IC t-stat (NW)']:+.2f}); "
    f"ARIMA {dir_tab.loc['ARIMA median', 'IC (Spearman)']:+.3f}, momentum {dir_tab.loc['1-month momentum', 'IC (Spearman)']:+.3f}.",
    f"Calibration: q10-q90 covers {c['Coverage q10-q90 (target 80%)']:.0%} (target 80%); "
    f"{c['Below q10 (target 10%)']:.0%} of months fell below q10 (target 10%).",
    f"Accuracy: pinball-loss skill vs random walk {c['Skill vs random walk']:+.1%} (DM t {c['DM t-stat (loss - RW)']:+.2f}); "
    f"ARIMA {cal_tab.loc['ARIMA', 'Skill vs random walk']:+.1%}.",
    f"Volatility: band width vs |realised| Spearman {c['Width vs |realised| (Spearman)']:+.2f} "
    f"(random walk {cal_tab.loc['Random walk', 'Width vs |realised| (Spearman)']:+.2f}).",
    f"Timing value ({LABEL[MAIN_RULE]}): excess return on - off {inf.get('Difference (on - off)', np.nan):+.2%}/month "
    f"(NW t {inf.get('t-stat (NW)', np.nan):+.2f}); beats {placebo_tab.loc[LABEL[MAIN_RULE], 'share of placebos beaten']:.0%} of random-timing placebos."
    if LABEL[MAIN_RULE] in placebo_tab.index else f"Timing value ({LABEL[MAIN_RULE]}): rule never switches, no placebo.",
    f"Backtest ({LABEL[MAIN_RULE]}, net): CAGR {s_main['CAGR']:.2%}, Sharpe {s_main['Sharpe']:.2f}, max DD {s_main['Max DD']:.1%}, "
    f"{s_main['Time in market']:.0%} in market | buy & hold {s_bh['CAGR']:.2%} / {s_bh['Sharpe']:.2f} / {s_bh['Max DD']:.1%} | "
    f"SMA{SMA_WINDOW} {s_sma['CAGR']:.2%} / {s_sma['Sharpe']:.2f} / {s_sma['Max DD']:.1%}.",
]
for line in lines:
    print("* " + line)

* Direction: TimesFM median hit rate 60.6% vs 63.7% for always-up (p = 0.90); it forecasts 'up' in 86% of months.
* Ranking: Spearman IC +0.048 (Newey-West t +0.94); ARIMA +0.024, momentum -0.027.
* Calibration: q10-q90 covers 74% (target 80%); 12% of months fell below q10 (target 10%).
* Accuracy: pinball-loss skill vs random walk +4.6% (DM t -2.66); ARIMA +0.9%.
* Volatility: band width vs |realised| Spearman +0.40 (random walk +0.27).
* Timing value (R1 TimesFM median): excess return on - off +0.71%/month (NW t +1.12); beats 86% of random-timing placebos.
* Backtest (R1 TimesFM median, net): CAGR 9.79%, Sharpe 0.50, max DD -44.1%, 81% in market | buy & hold 9.96% / 0.48 / -55.4% | SMA200 9.77% / 0.63 / -25.4%.


## 9. Caveats

* **Pre-training contamination** -- TimesFM was pre-trained on a large public time-series corpus that may include S&P 500 / SPY prices. Every forecast before the weights were published (`MODEL_RELEASE`) can therefore be partly "remembered" rather than predicted, which would flatter the in-sample results. Only the months after the release are clean, and there are few of them.
* **Price-level input** -- the model sees a price level series (as in the app's tool). A near-random-walk series gives a forecast median close to the last price, so small median moves drive R1. The decile *spread* is usually more informative than the median.
* **Approximate horizon** -- the step $k$ counts Mon-Fri business days to the month-end and ignores exchange holidays, so the forecast target can end 0-2 trading days after the real next decision.
* **Quantiles** -- TimesFM returns deciles only. R2's probability is a linear interpolation between them, and anything beyond q10 / q90 is capped at 5% / 95%.
* **Dividend adjustment** -- `Adj Close` is rescaled retroactively by later dividends. This changes the level but not the returns, and TimesFM normalises its inputs.
* **Data, T-bills, costs, gap risk** -- as in the SMA backtest: Yahoo Finance via `yfinance`, `^IRX` accrual, a flat `COST_BPS` per switch, overnight gap between the decision close and the fill.
* **Overfitting** -- four rules, two threshold sweeps and several comparisons are many looks at one sample. Treat any single good number with suspicion; the placebo and the regime inference are the guard rails.
* **Compute** -- a CPU forecast takes ~3 s, so the walk-forward is cached under `.cache/notebooks/timesfm/`. `REFRESH_FORECASTS = True` recomputes everything.

## 10. Interpretation

*Written from the run of 19 Oct 2026 (355 completed months, 1997-2026). Section 8 recomputes the numbers on every run, so check them there if the data has moved on.*

1. **TimesFM does not forecast the *direction* of next month better than "always up".** Its median is positive in ~86% of months and right ~61% of the time, below the ~64% of always predicting up. The rank correlation with realised returns is small (~+0.05) and not significant. ARIMA and 1-month momentum are no better.
2. **It does forecast *risk*.** The decile forecast beats the random-walk cone on pinball loss (~+5% skill, Diebold-Mariano t ~ -2.7), and wider forecast bands precede bigger moves (Spearman ~+0.40 vs ~+0.27 for trailing volatility). The bands are too narrow, though: q10-q90 catches ~74% of outcomes instead of 80%, with both tails under-estimated.
3. **R1 acts like a slow trend filter.** With ~4 years of daily context the median extrapolates the multi-year trend. It went to T-bills during 2001-02 (helpful) but stayed out through much of the 2003-06 recovery (costly). Net of costs it roughly matches buy & hold on CAGR, with a smaller max drawdown and a marginally higher Sharpe. It beats ~86% of random-timing placebos (short of a 95% bar), and the on-minus-off excess return is not significant.
4. **SMA200 monthly is still the better timing rule** on the same window: higher Sharpe and about half the drawdown. R4 (TimesFM **and** SMA200) mostly reproduces the SMA rule with extra switches, so TimesFM adds nothing on top of the trend filter. R3 is counter-productive: months with a fat forecast left tail were followed by *higher* average returns, because high-volatility periods tend to be followed by rebounds.
5. **Contamination cannot be ruled out.** There are only ~12 post-release months, far too few to separate real skill from pre-training memory. Any genuine edge is therefore at most what the pre-release sample shows.

**Bottom line.** As a buy/sell timing signal for the S&P 500, TimesFM's median is not worth trading on. Its value lies in the *spread* of the forecast (risk sizing), not the median. Untested follow-ups:
* volatility targeting with the forecast band width as the risk estimate;
* feeding log returns instead of price levels;
* using TimesFM as a risk overlay on the SMA200 rule rather than as a direction signal.